# SAS to Databricks: Credit Pricing Regression with Feature Engineering

This notebook uses financial credit-pricing data to show how SAS regression ideas map to Databricks and PySpark ML.

You will learn how to:
- Create realistic credit and loan pricing data
- Visualize credit risk patterns with Matplotlib
- Train a multiple linear regression model
- Add polynomial features to capture curved pricing behavior
- Compare train and test performance


## 1. Import Libraries

The code uses clear Python names and keeps explanations in markdown so code cells stay clean.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression

warnings.filterwarnings("ignore")

print("Libraries are ready")
print(f"Spark version: {spark.version}")


## 2. Create Credit Pricing Data

The target column is `interest_rate_percent`. It is based on common credit pricing signals such as credit score, debt-to-income, utilization, delinquencies, loan term, and prior default history.

The relationship is intentionally curved: very low credit scores receive a stronger rate increase than mid-range scores. This makes polynomial regression useful later.


In [ ]:
np.random.seed(42)
row_count = 800

credit_score = np.random.randint(520, 851, row_count)
annual_income = np.random.normal(85000, 30000, row_count).clip(25000, 220000)
loan_amount = np.random.uniform(5000, 75000, row_count)
debt_to_income = np.random.uniform(0.05, 0.55, row_count)
credit_utilization = np.random.uniform(0.05, 0.95, row_count)
delinquencies = np.random.poisson(0.5, row_count).clip(0, 5)
loan_term_months = np.random.choice([36, 48, 60, 72], row_count)
prior_default = np.random.choice([0, 1], row_count, p=[0.88, 0.12])

score_risk = np.maximum((720 - credit_score) / 100, 0)
amount_to_income = loan_amount / annual_income

interest_rate_percent = (
    4.5
    + 1.8 * score_risk
    + 2.2 * score_risk**2
    + 6.0 * debt_to_income
    + 3.0 * credit_utilization
    + 0.45 * delinquencies
    + 1.2 * prior_default
    + 0.02 * (loan_term_months - 36)
    + 2.0 * amount_to_income
    + np.random.normal(0, 0.6, row_count)
).clip(3.0, 28.0)

credit_pricing_data = pd.DataFrame({
    "credit_score": credit_score,
    "annual_income": annual_income,
    "loan_amount": loan_amount,
    "debt_to_income": debt_to_income,
    "credit_utilization": credit_utilization,
    "delinquencies": delinquencies,
    "loan_term_months": loan_term_months,
    "prior_default": prior_default,
    "interest_rate_percent": interest_rate_percent,
})

credit_pricing = spark.createDataFrame(credit_pricing_data)

print(f"Rows: {credit_pricing.count()}")
credit_pricing.show(5)


## 3. SAS Reference: Credit Pricing Data

SAS would usually create or load these same financial features into one dataset before modeling.

```sas
DATA credit_pricing;
  DO customer_id = 1 TO 800;
    credit_score = 520 + FLOOR(RAND('UNIFORM') * 331);
    annual_income = MAX(25000, RAND('NORMAL', 85000, 30000));
    loan_amount = 5000 + RAND('UNIFORM') * 70000;
    debt_to_income = 0.05 + RAND('UNIFORM') * 0.50;
    credit_utilization = 0.05 + RAND('UNIFORM') * 0.90;
    delinquencies = RAND('POISSON', 0.5);
    loan_term_months = 36;
    prior_default = (RAND('UNIFORM') > 0.88);
    interest_rate_percent = 4.5 + 6*debt_to_income + 3*credit_utilization;
    OUTPUT;
  END;
RUN;
```


## 4. Visualize Credit Pricing Relationships

Matplotlib helps learners see the pricing pattern before modeling. Lower credit scores and higher debt-to-income values generally create higher interest rates.


In [ ]:
plot_data = credit_pricing.select(
    "credit_score",
    "debt_to_income",
    "interest_rate_percent",
).limit(400).toPandas()

plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    plot_data["credit_score"],
    plot_data["interest_rate_percent"],
    c=plot_data["debt_to_income"],
    alpha=0.75,
)
plt.title("Interest Rate by Credit Score and Debt-to-Income")
plt.xlabel("Credit Score")
plt.ylabel("Interest Rate (%)")
plt.colorbar(scatter, label="Debt-to-Income")
plt.grid(True)
plt.show()


## 5. Split Data into Train and Test Sets

Train data teaches the model. Test data checks how well the model works on new credit applications.


In [ ]:
train_data, test_data = credit_pricing.randomSplit([0.7, 0.3], seed=42)

feature_columns = [
    "credit_score",
    "annual_income",
    "loan_amount",
    "debt_to_income",
    "credit_utilization",
    "delinquencies",
    "loan_term_months",
    "prior_default",
]
target_column = "interest_rate_percent"

train_count = train_data.count()
test_count = test_data.count()
total_count = train_count + test_count

print(f"Train rows: {train_count} ({train_count / total_count:.1%})")
print(f"Test rows: {test_count} ({test_count / total_count:.1%})")


## 6. SAS Reference: Train-Test Split

SAS can create a train-test split with a random value. PySpark uses `randomSplit`.

```sas
DATA credit_split;
  SET credit_pricing;
  random_value = RAND('UNIFORM');
  IF random_value < 0.70 THEN split = 'train';
  ELSE split = 'test';
RUN;

DATA train_data test_data;
  SET credit_split;
  IF split = 'train' THEN OUTPUT train_data;
  ELSE OUTPUT test_data;
RUN;
```


## 7. Build a Linear Pricing Pipeline

A PySpark pipeline keeps feature assembly, scaling, and regression together. This is easier to reuse than manually repeating each step.


In [ ]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=False, withStd=True)
regression = LinearRegression(featuresCol="scaled_features", labelCol=target_column, regParam=0.01)

linear_pipeline = Pipeline(stages=[assembler, scaler, regression])

print("Training linear pricing pipeline...")
linear_model = linear_pipeline.fit(train_data)
print("Training complete")

regression_stage = linear_model.stages[-1]

print("\nModel coefficients")
for column_name, coefficient in zip(feature_columns, regression_stage.coefficients):
    print(f"{column_name:<20} {coefficient:>12,.4f}")
print(f"{'intercept':<20} {regression_stage.intercept:>12,.4f}")


## 8. SAS Reference: Scaling and Linear Pricing Model

SAS often standardizes variables with `PROC STANDARD`, then trains with `PROC REG`.

```sas
PROC STANDARD DATA=train_data MEAN=0 STD=1 OUT=train_scaled;
  VAR credit_score annual_income loan_amount debt_to_income credit_utilization
      delinquencies loan_term_months prior_default;
RUN;

PROC REG DATA=train_scaled;
  MODEL interest_rate_percent = credit_score annual_income loan_amount debt_to_income
      credit_utilization delinquencies loan_term_months prior_default;
RUN;
QUIT;
```


## 9. Evaluate the Linear Model

RMSE and MAE are error values, so lower is better. R2 is a fit score, so higher is better.


In [ ]:
train_predictions = linear_model.transform(train_data)
test_predictions = linear_model.transform(test_data)

rmse_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="mae")
r2_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="r2")

train_rmse = rmse_evaluator.evaluate(train_predictions)
test_rmse = rmse_evaluator.evaluate(test_predictions)
train_mae = mae_evaluator.evaluate(train_predictions)
test_mae = mae_evaluator.evaluate(test_predictions)
train_r2 = r2_evaluator.evaluate(train_predictions)
test_r2 = r2_evaluator.evaluate(test_predictions)

print(f"{'Metric':<10} {'Train':>15} {'Test':>15}")
print("-" * 42)
print(f"{'RMSE':<10} {train_rmse:>15.4f} {test_rmse:>15.4f}")
print(f"{'MAE':<10} {train_mae:>15.4f} {test_mae:>15.4f}")
print(f"{'R2':<10} {train_r2:>15.4f} {test_r2:>15.4f}")


## 10. Visualize Actual vs Predicted Rates

Dots close to the red diagonal line are better predictions.


In [ ]:
actual_vs_predicted = test_predictions.select(target_column, "prediction").toPandas()

min_rate = min(actual_vs_predicted[target_column].min(), actual_vs_predicted["prediction"].min())
max_rate = max(actual_vs_predicted[target_column].max(), actual_vs_predicted["prediction"].max())

plt.figure(figsize=(6, 6))
plt.scatter(actual_vs_predicted[target_column], actual_vs_predicted["prediction"], alpha=0.65)
plt.plot([min_rate, max_rate], [min_rate, max_rate], color="red", label="Perfect prediction")
plt.title("Actual vs Predicted Interest Rates")
plt.xlabel("Actual Interest Rate (%)")
plt.ylabel("Predicted Interest Rate (%)")
plt.legend()
plt.grid(True)
plt.show()


## 11. Add Polynomial Features for Curved Pricing

Credit pricing often changes in a curved way. Moving from a 780 score to 740 may have a smaller pricing impact than moving from 620 to 580. Polynomial features help model that curve.


In [ ]:
polynomial_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
polynomial_expansion = PolynomialExpansion(degree=2, inputCol="features", outputCol="polynomial_features")
polynomial_scaler = StandardScaler(inputCol="polynomial_features", outputCol="scaled_polynomial_features", withMean=False, withStd=True)
polynomial_regression = LinearRegression(featuresCol="scaled_polynomial_features", labelCol=target_column, regParam=0.01)

polynomial_pipeline = Pipeline(stages=[
    polynomial_assembler,
    polynomial_expansion,
    polynomial_scaler,
    polynomial_regression,
])

print("Training polynomial pricing pipeline...")
polynomial_model = polynomial_pipeline.fit(train_data)
print("Training complete")

polynomial_train_predictions = polynomial_model.transform(train_data)
polynomial_test_predictions = polynomial_model.transform(test_data)

polynomial_train_r2 = r2_evaluator.evaluate(polynomial_train_predictions)
polynomial_test_r2 = r2_evaluator.evaluate(polynomial_test_predictions)
polynomial_test_rmse = rmse_evaluator.evaluate(polynomial_test_predictions)

print(f"{'Model':<18} {'Train R2':>12} {'Test R2':>12} {'Test RMSE':>15}")
print("-" * 62)
print(f"{'Linear':<18} {train_r2:>12.4f} {test_r2:>12.4f} {test_rmse:>15.4f}")
print(f"{'Polynomial':<18} {polynomial_train_r2:>12.4f} {polynomial_test_r2:>12.4f} {polynomial_test_rmse:>15.4f}")


## 12. SAS Reference: Polynomial Credit Pricing

SAS can create a squared risk variable before regression. PySpark `PolynomialExpansion` creates squared and interaction terms automatically.

```sas
DATA train_poly;
  SET train_data;
  score_risk = MAX((720 - credit_score) / 100, 0);
  score_risk_sq = score_risk * score_risk;
RUN;

PROC REG DATA=train_poly;
  MODEL interest_rate_percent = score_risk score_risk_sq debt_to_income
      credit_utilization delinquencies prior_default;
RUN;
QUIT;
```


## 13. Visualize the Polynomial Pricing Curve

This chart holds most borrower values fixed and changes only the credit score. It shows how the polynomial model can learn a curved pricing pattern.


In [ ]:
credit_score_grid = pd.DataFrame({
    "credit_score": np.arange(520, 851, 5),
    "annual_income": 85000,
    "loan_amount": 30000,
    "debt_to_income": 0.28,
    "credit_utilization": 0.35,
    "delinquencies": 0,
    "loan_term_months": 60,
    "prior_default": 0,
})

curve_data = spark.createDataFrame(credit_score_grid)
linear_curve = linear_model.transform(curve_data).select("credit_score", "prediction").toPandas()
polynomial_curve = polynomial_model.transform(curve_data).select("credit_score", "prediction").toPandas()

plt.figure(figsize=(9, 5))
plt.plot(linear_curve["credit_score"], linear_curve["prediction"], label="Linear model")
plt.plot(polynomial_curve["credit_score"], polynomial_curve["prediction"], label="Polynomial model")
plt.title("Credit Score Pricing Curve")
plt.xlabel("Credit Score")
plt.ylabel("Predicted Interest Rate (%)")
plt.legend()
plt.grid(True)
plt.show()


## 14. Visualize Train and Test R2 Scores

R2 is a fit score, so taller bars are better. If train R2 is much higher than test R2, the model may be overfitting.


In [ ]:
model_names = ["Linear Train", "Linear Test", "Polynomial Train", "Polynomial Test"]
r2_scores = [train_r2, test_r2, polynomial_train_r2, polynomial_test_r2]

plt.figure(figsize=(9, 5))
plt.bar(model_names, r2_scores)
plt.title("Credit Pricing R2 Score Comparison")
plt.ylabel("R2 Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.show()


## 15. Migration Notes

| Step | SAS | Databricks |
| --- | --- | --- |
| Credit data | `DATA` step or imported table | Spark DataFrame |
| Train-test split | Random value and `IF` logic | `randomSplit` |
| Standardization | `PROC STANDARD` | `StandardScaler` |
| Linear pricing model | `PROC REG` | `LinearRegression` |
| Curved pricing terms | Manual squared columns | `PolynomialExpansion` |
| Score new applications | `PROC SCORE` or `OUTPUT OUT=` | `model.transform()` |
| Visual review | `PROC SGPLOT` | Matplotlib |

Start with the linear model as a baseline. Use polynomial features when the pricing relationship is visibly curved or when test metrics improve enough to justify the added complexity.
